In [1]:
#!pip uninstall -y transformers
#!pip install git+https://github.com/huggingface/transformers.git@fix/lerobot_openpi

In [2]:
from datetime import datetime
import random
import numpy as np
import os
import torch
import json
from PIL import Image
from src.env.env_clr import RILAB_OMY_ENV
from torchvision import transforms

from lerobot.policies.pi0.modeling_pi0 import PI0Policy
from lerobot.processor import PolicyAction, PolicyProcessorPipeline
from lerobot.processor.converters import (
    batch_to_transition,
    policy_action_to_transition,
    transition_to_batch,
    transition_to_policy_action,
)
from lerobot.utils.constants import POLICY_POSTPROCESSOR_DEFAULT_NAME, POLICY_PREPROCESSOR_DEFAULT_NAME

import glfw

/opt/miniconda3/envs/mujoco2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import configparser
config = configparser.ConfigParser()
config.read("experiment-pi0.cfg")
exp = config["experiment"]

DATASET_ROOT = exp["DATASET_ROOT"]
DATASET_REPO = exp["DATASET_REPO"]
POLICY_REPO = exp["POLICY_REPO"]
OUTPUT_DIR = exp["OUTPUT_DIR"]
JOB_NAME = exp["JOB_NAME"] + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
MAX_TRAIN_STEPS = int(exp["MAX_TRAIN_STEPS"])
CHUNK_SIZE = int(exp["CHUNK_SIZE"])
ACTION_STEPS = int(exp["ACTION_STEPS"])
BATCH_SIZE = int(exp["BATCH_SIZE"])

In [4]:
'''
Load environment configuration and initialize environments
'''
# Evaluation Configuration
TEST_EPISODES = 10 #@param {"type":"integer"}
MAX_EPISODE_STEPS = 30_000 #@param {"type":"string"}
#TASK="Pick up the bag by the red handle and put it on the blue bench in front of the cubbies." #@param {"type":"string"}
TASK=exp["TASK"]

## Load Model

In [5]:
'''
Meta data is for loading dataset statistics and feature information
'''
#repo_id_or_path = 'Jeongeun/tutorial_v2_pi05' # Use this for loading pretrained model from the hub
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

policy = PI0Policy.from_pretrained(POLICY_REPO)
_ = policy.to(device)



The PI0 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi


Loading model from: gimarchetti/clr-experiment-pi0


✓ Loaded state dict from model.safetensors
	Missing key(s) in state_dict: "model.paligemma_with_expert.paligemma.model.language_model.embed_tokens.weight". 


**Note**: If you want to change number of actions in chunk to be executed, please change:  

```policy.config.n_action_steps = YOUR_DESIRED_NUMBER```

In [6]:
# Set the number of action steps to run in the environment for one invocation of the policy
policy.config.n_action_steps = 15

In [7]:
# Check normalization stats dimension
preprocessor = PolicyProcessorPipeline.from_pretrained(
                pretrained_model_name_or_path=POLICY_REPO,
                config_filename= f"{POLICY_PREPROCESSOR_DEFAULT_NAME}.json",
                overrides={"device_processor": {"device": device}},
                to_transition=batch_to_transition,
                to_output=transition_to_batch,
            )

for step in preprocessor.steps:
    if hasattr(step, "stats"):
        if "observation.state" in step.stats:
            print(f"Stats dimension for observation.state: {step.stats['observation.state']['mean'].shape}")

postprocessor = PolicyProcessorPipeline.from_pretrained(
                pretrained_model_name_or_path=POLICY_REPO,
                config_filename= f"{POLICY_POSTPROCESSOR_DEFAULT_NAME}.json",
                overrides={"device_processor": {"device": device}},
                to_transition=policy_action_to_transition,
                to_output=transition_to_policy_action,
            )

Stats dimension for observation.state: (7,)


In [8]:
batch = {
    'observation.state': np.zeros((1, 7), dtype=np.float32),
    'observation.image': np.zeros((1, 3, 448, 448), dtype=np.float32),
    'observation.wrist_image': np.zeros((1, 3, 448, 448), dtype=np.float32),
    'observation.left_scene_image': np.zeros((1, 3, 448, 448), dtype=np.float32),
    'observation.right_scene_image': np.zeros((1, 3, 448, 448), dtype=np.float32),
    'task': [TASK]
}
batch = preprocessor(batch)  # to initialize the processors
batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

_ = policy.select_action(batch)  # to initialize the model

## Load Environment

In [9]:
config_file_path = './configs/train_clr.json'
with open(config_file_path) as f:
    env_conf = json.load(f)
omy_env = RILAB_OMY_ENV(cfg=env_conf, seed=0, 
                        action_type='joint', 
                        obs_type='joint_pos',
                        vis_mode = 'teleop')


-----------------------------------------------------------------------------
name:[tabletop_env] dt:[0.002] HZ:[500]
 n_qpos:[40] n_qvel:[39] n_qacc:[39] n_ctrl:[9]
 integrator:[IMPLICITFAST]

n_body:[35]
 [0/35] [world] mass:[0.00]kg
 [1/35] [vention_rail_carriage] mass:[22.53]kg
 [2/35] [ewellix_lift_higher_link] mass:[19.17]kg
 [3/35] [ewellix_lift_middle_link] mass:[15.59]kg
 [4/35] [shoulder_link] mass:[7.37]kg
 [5/35] [upper_arm_link] mass:[13.05]kg
 [6/35] [forearm_link] mass:[3.99]kg
 [7/35] [wrist_1_link] mass:[2.10]kg
 [8/35] [wrist_2_link] mass:[1.98]kg
 [9/35] [wrist_3_link] mass:[1.56]kg
 [10/35] [finger_1_link] mass:[0.05]kg
 [11/35] [finger_2_link] mass:[0.05]kg
 [12/35] [door] mass:[0.30]kg
 [13/35] [right_latch_pull] mass:[0.10]kg
 [14/35] [left_latch_pull] mass:[0.10]kg
 [15/35] [latch_lock] mass:[0.10]kg
 [16/35] [lorge/hatch_face] mass:[19.28]kg
 [17/35] [lorge/external_rotary_wheel] mass:[0.41]kg
 [18/35] [lorge/external_rotary_handle] mass:[0.03]kg
 [19/35] [lor

In [10]:
def get_default_transform():
    """
    Returns a torchvision transform that:
     Converts to a FloatTensor and scales pixel values [0,255] -> [0.0,1.0]
    """
    return transforms.Compose([
        transforms.ToTensor(),  # PIL [0–255] -> FloatTensor [0.0–1.0], shape C×H×W
    ])
IMG_TRANSFORM = get_default_transform()

## Evaluate

In [11]:
'''
Run one evaluation episode
'''
def run_one_episode():
    omy_env.reset(leader_pose=True)
    policy.reset()
    observation = omy_env.get_observation()
    omy_env.env.tick = 0
    success = False
    while omy_env.env.is_viewer_alive() and omy_env.env.tick < MAX_EPISODE_STEPS:
        omy_env.step_env()
        if omy_env.env.loop_every(HZ = 20):
            success = omy_env.check_success()
            if success: break
            if omy_env.env.is_key_pressed_once(glfw.KEY_Z):
                break  # for debugging: press 'z' to end the episode
            
            agent_image, wrist_image, left_scene_image, right_scene_image = omy_env.grab_image()
            
            frame = {
                "observation.state": observation[:7].astype(np.float32),
                'task': [TASK]
            }
            
            agent_image = Image.fromarray(agent_image)
            wrist_image = Image.fromarray(wrist_image)
            left_scene_image = Image.fromarray(left_scene_image)
            right_scene_image = Image.fromarray(right_scene_image)
            
            agent_image = agent_image.resize((448, 448))
            wrist_image = wrist_image.resize((448, 448))
            left_scene_image = left_scene_image.resize((448, 448))
            right_scene_image = right_scene_image.resize((448, 448))
            
            agent_image = IMG_TRANSFORM(agent_image)
            wrist_image = IMG_TRANSFORM(wrist_image)
            left_scene_image = IMG_TRANSFORM(left_scene_image)
            right_scene_image = IMG_TRANSFORM(right_scene_image)
            
            frame["observation.image"] = agent_image
            frame["observation.wrist_image"] = wrist_image
            frame["observation.left_scene_image"] = left_scene_image
            frame["observation.right_scene_image"] = right_scene_image
            
            # numpy to torch
            frame = {k: torch.tensor(v, dtype=torch.float32).unsqueeze(0) if isinstance(v, np.ndarray) else v for k, v in frame.items()}
            # pre-process the frame
            frame = preprocessor(frame)
            # move to device
            frame = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in frame.items()}
            # select action
            action = policy.select_action(frame)
            # post-process the action
            action = postprocessor(action)
            action = action.squeeze(0).cpu().numpy()
            observation = omy_env.step(action, gripper_mode='binary')
            omy_env.render()
    return success

In [12]:
'''
Run evaluation over multiple episodes
'''
results = []
for episode in range(TEST_EPISODES):
    success = run_one_episode()
    results.append(success)
    print(f"Episode {episode+1}/{TEST_EPISODES} - Success: {success}")
omy_env.env.close_viewer()
# log average success rate
avg_success = sum(results) / len(results)
print(f"Average Success Rate over {TEST_EPISODES} episodes: {avg_success*100:.2f}%")


DONE INITIALIZATION
Episode 1/10 - Success: True
DONE INITIALIZATION
Episode 2/10 - Success: True
DONE INITIALIZATION
Episode 3/10 - Success: True
DONE INITIALIZATION
Episode 4/10 - Success: True
DONE INITIALIZATION
Episode 5/10 - Success: True
DONE INITIALIZATION
Episode 6/10 - Success: True
DONE INITIALIZATION
Episode 7/10 - Success: False
DONE INITIALIZATION
Episode 8/10 - Success: False
DONE INITIALIZATION
Episode 9/10 - Success: False
DONE INITIALIZATION
Episode 10/10 - Success: False
Average Success Rate over 10 episodes: 60.00%
